# 02 — Instruments cache and symbol resolution

Every Kite endpoint needs an `instrument_token`, not a human symbol. `InstrumentsCache` fetches Kite's daily dump, persists it per-day as Parquet, and resolves symbols in O(log N).

This notebook uses a **fake** instruments list so no network is needed.

In [ ]:
import datetime as dt
from ml4t.india.kite.fake import FakeKiteClient
from ml4t.india.kite.instruments import InstrumentsCache

sdk = FakeKiteClient()
sdk.set_instruments([
    {'instrument_token': 738561, 'exchange_token': 2885, 'tradingsymbol': 'RELIANCE',
     'name': 'RELIANCE INDS', 'last_price': 2500.0, 'expiry': None, 'strike': 0,
     'tick_size': 0.05, 'lot_size': 1, 'instrument_type': 'EQ',
     'segment': 'NSE', 'exchange': 'NSE'},
    {'instrument_token': 2953217, 'exchange_token': 11536, 'tradingsymbol': 'TCS',
     'name': 'TATA CONSULTANCY', 'last_price': 3800.0, 'expiry': None, 'strike': 0,
     'tick_size': 0.05, 'lot_size': 1, 'instrument_type': 'EQ',
     'segment': 'NSE', 'exchange': 'NSE'},
])

# Build a cache and feed it the fake dump manually.
import tempfile, pathlib
cache = InstrumentsCache(cache_dir=pathlib.Path(tempfile.mkdtemp()))


Resolve a symbol:

In [ ]:
# In real usage: cache.refresh(sdk) fetches + writes Parquet.
# For the demo we seed the in-memory frame directly:
import polars as pl
cache._frame = pl.DataFrame(sdk.instruments())  # type: ignore[attr-defined]
meta = cache.resolve('NSE:RELIANCE')
print(meta.tradingsymbol, meta.instrument_token, meta.lot_size)
